In [ ]:
# 1
import pandas as pd
import glob
import os
from collections import Counter
from IPython.display import display
import ipaddress
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
import ipywidgets as widgets
import re


Flow Log Signals: Processes the output of the Ml based hunting notebooks and assembles detection lattices. Part of the DUNE project (https://github.com/opendr-io/dune) and useful for hunting threats that are resistant to conventional detection or just hunting at great scale.

In [ ]:
# 2 - settings. DATA_PATH: the folder of detector output CSVs. PYOD_MIN_RANK: keep only PyOD
# sessions flagged by at least this many of its models (rank); None keeps them all.
DATA_PATH     = r'output'
PYOD_MIN_RANK = 6
INCLUDE_NORTH_SOUTH = True
INCLUDE_EAST_WEST   = True
INCLUDE_ACCEPT      = True
INCLUDE_REJECT      = False

In [ ]:
# 3 - event ports for specification based detections: the lines of conf/tcp-ports.md and
# conf/udp-ports.md whose event column is true, one row per port ("67 / 68" gives two rows),
# into the data frame event_ports: protocol (6 TCP, 17 UDP), port, service, description.
# Lines are "port<TAB>service<TAB>description<TAB>event", as the pre-processor reads them;
# headings and header lines are skipped. A file that's missing is warned about and skipped.

CONF_DIR    = 'conf'
EVENT_FILES = {6: 'tcp-ports.md', 17: 'udp-ports.md'}

_rows = []
for _proto, _fn in EVENT_FILES.items():
    _path = os.path.join(CONF_DIR, _fn)
    if not os.path.exists(_path):
        print(f"!! {_path} not found, skipped")
        continue
    with open(_path, encoding='utf-8') as _f:
        for _no, _ln in enumerate(_f, 1):
            if not _ln.strip()[:1].isdigit():
                continue
            _parts = [p.strip() for p in re.split(r'\t+|\s{2,}', _ln.strip())]
            if len(_parts) < 4:
                print(f"!! {_fn} line {_no} has no event column, skipped: {_ln.strip()!r}")
                continue
            if _parts[3].lower() != 'true':
                continue
            for _p in _parts[0].split('/'):
                _rows.append({'protocol': _proto, 'port': int(_p), 'service': _parts[1],
                              'description': _parts[2]})

event_ports = pd.DataFrame(_rows, columns=['protocol', 'port', 'service', 'description'])
print(f"{len(event_ports)} event ports: "
      f"{(event_ports['protocol'] == 6).sum()} TCP, {(event_ports['protocol'] == 17).sum()} UDP")
event_ports

In [ ]:
# 4 - ingest signal data: the anomaly outputs that pyod-vflows (cell 28), kmeans-mark-3 and
# flows-dbscan (cell 14) write to DATA_PATH, one CSV per detector per run, named
# <detector>-output-<account>-<month>-<day>-<year>.csv. Each row is one flagged flow session
# (session_id), and its "type" field says which detector flagged it.
# LATEST_ONLY = True reads only the newest file per detector and account (by the date in the
# name), so runs on different days don't count the same session twice; False reads them all.
# CSVs not named that way are skipped (and listed).

LATEST_ONLY = True

_NAME = re.compile(r'^(?P<det>[a-z]+)-output-(?P<acct>[\d+]+)-(?P<mon>[a-z]+)-(?P<day>\d+)-(?P<yr>\d{4})\.csv$')

# column types. CSV doesn't keep them, so: IDs are read as text (an account id can start with
# 0, which reading it as a number would drop), and the count / number columns become Int64
# (whole numbers that allow blanks) -- otherwise a column one detector doesn't write turns
# float for the whole table. Columns not listed keep what pandas reads.
STRING_COLS = ['account-id', 'interface-id', 'session_id']
INT_COLS    = ['rank', 'duration', 'bytes_per_packet', 'bytes', 'packets', 'records', 'start', 'end',
               'version', 'srcport', 'dstport', 'protocol', 'src-int', 'dst-int', 'cluster',
               'outlier_kmeans']

def load_file(path):
    df = pd.read_csv(path, low_memory=False, dtype={c: 'string' for c in STRING_COLS})
    for col in ('type', 'session_id'):
        if col not in df.columns:
            raise ValueError(f"{path} has no '{col}' field -- re-run the output cell of the notebook that wrote it")
    df['source_file'] = os.path.basename(path)
    return df

found, skipped = [], []
for f in sorted(glob.glob(os.path.join(DATA_PATH, '*.csv'))):
    m = _NAME.match(os.path.basename(f))
    if not m:
        skipped.append(os.path.basename(f))
        continue
    found.append(dict(detector=m['det'], account=m['acct'], file=os.path.basename(f), path=f,
                      date=pd.to_datetime(f"{m['mon']} {m['day']} {m['yr']}", format='%B %d %Y')))
if skipped:
    print(f"skipped {len(skipped)} CSVs not named <detector>-output-<account>-<month>-<day>-<year>.csv: {skipped}")
if not found:
    raise FileNotFoundError(f"no <detector>-output-...csv files in {os.path.abspath(DATA_PATH)}")
files = pd.DataFrame(found).sort_values(['detector', 'account', 'date'])
if LATEST_ONLY:
    files = files.groupby(['detector', 'account']).tail(1)
print(f"reading {len(files)} files from {os.path.abspath(DATA_PATH)}:")
for r in files.itertuples():
    print(f"  {r.file}")

signals_raw = pd.concat((load_file(f) for f in files['path']), ignore_index=True)
for _c in [c for c in INT_COLS if c in signals_raw.columns]:
    signals_raw[_c] = pd.to_numeric(signals_raw[_c], errors='coerce').round().astype('Int64')

# PyOD rank: how many of its models flagged the session (higher = more models agreed)
_pyod = signals_raw['type'].eq('pyod anomaly detection')
print()
if _pyod.any() and 'rank' in signals_raw.columns:
    _rc = pd.to_numeric(signals_raw.loc[_pyod, 'rank'], errors='coerce').astype('Int64').value_counts(dropna=False).sort_index()
    print(f"PyOD rank counts ({int(_pyod.sum()):,} sessions):")
    print(_rc.rename('sessions').to_frame().rename_axis('rank').to_string(formatters={'sessions': '{:,}'.format}))
else:
    print("no PyOD rows with a rank field loaded")

# keep PyOD sessions with rank >= PYOD_MIN_RANK (cell 2); other detectors' rows are untouched
if PYOD_MIN_RANK is not None and _pyod.any():
    _low = _pyod & ~pd.to_numeric(signals_raw['rank'], errors='coerce').ge(PYOD_MIN_RANK).fillna(False)
    signals_raw = signals_raw[~_low.to_numpy(bool)].reset_index(drop=True)
    print(f"PYOD_MIN_RANK = {PYOD_MIN_RANK}: kept {int(_pyod.sum() - _low.sum()):,} PyOD sessions, "
          f"left out {int(_low.sum()):,}")

(
    signals_raw.groupby(['type', 'action'], dropna=False)
    .agg(sessions=('session_id', 'size'), ENIs=('interface-id', 'nunique'))
    .reset_index()
    .style.hide(axis='index')
    .format({'sessions': '{:,}'})
    .set_table_styles([{'selector': 'th', 'props': 'text-align: left; padding: 4px 12px;'},
                       {'selector': 'td', 'props': 'padding: 4px 12px;'}])
    .background_gradient(subset=['sessions'], cmap='Blues')
)

In [ ]:
# 5
signals_raw.head()

In [ ]:
# 6 - construct detection lattices

# corroborated: sessions (session_id) flagged by more than one detector
types_per_session = signals_raw.groupby('session_id')['type'].nunique()
duplicate_ids = types_per_session[types_per_session > 1].index
corroborated = (
    signals_raw[signals_raw['session_id'].isin(duplicate_ids)]
    .drop(columns=['outlier_kmeans_score', 'rank'], errors='ignore')
    .assign(type='Signal Fusion: Model Convergence')
    .drop_duplicates()
    .sort_values('session_id')
    .reset_index(drop=True)
)

# all_detections tuple
all_detections = tuple(
    (row['session_id'], row['type'])
    for df in [corroborated]
    for _, row in df[['session_id', 'type']].drop_duplicates().iterrows()
)

# signals_processed
signals_processed = pd.concat([
    signals_raw.drop(columns=['rank', 'outlier_kmeans_score'], errors='ignore'),
    corroborated,
], ignore_index=True).drop_duplicates()

# per-ENI (interface-id) breakdown with totals
pivot = signals_processed.pivot_table(
    index='interface-id',
    columns='type',
    aggfunc='size',
    fill_value=0
)
pd.concat([pivot, pivot.sum().rename('Total').to_frame().T])

In [ ]:
# 7 - search space reduction by tag-direction and action.
# INCLUDE_NORTH_SOUTH keeps sessions between an ENI and the internet, INCLUDE_EAST_WEST keeps
# private-to-private sessions. INCLUDE_ACCEPT keeps sessions that got through, INCLUDE_REJECT
# keeps sessions blocked by the security group / NACL. Rows with no value in a field are left
# out unless both of that field's settings are True.

def _pick(col, choices):
    keep = [v for v, on in choices if on]
    if not keep:
        raise ValueError(f"cell 7: set at least one {col} setting to True")
    m = signals_processed[col].isin(keep).fillna(False).to_numpy(bool)
    if len(keep) == len(choices):
        m |= signals_processed[col].isna().to_numpy(bool)
    return m, keep

_mDir, _dirs = _pick('tag-direction', [('north-south', INCLUDE_NORTH_SOUTH), ('east-west', INCLUDE_EAST_WEST)])
_mAct, _acts = _pick('action', [('ACCEPT', INCLUDE_ACCEPT), ('REJECT', INCLUDE_REJECT)])
sp = signals_processed[_mDir & _mAct]
print(f"kept {len(sp):,} rows ({', '.join(_dirs)}; {', '.join(_acts)}), "
      f"left out {len(signals_processed) - len(sp):,}")

In [ ]:
# 8 - ENI (interface-id) filters: keep / drop ENIs whose id contains this text
ENI_INCLUDE = ''
ENI_EXCLUDE = ''

if ENI_INCLUDE:
    sp = sp[sp['interface-id'].astype(str).str.contains(ENI_INCLUDE, case=False, na=False)]
if ENI_EXCLUDE:
    sp = sp[~sp['interface-id'].astype(str).str.contains(ENI_EXCLUDE, case=False, na=False)]

In [ ]:
# 9 - compute scores
# keys are the "type" values: the three detectors' (as written in their output files), then
# the ones cell 6 adds. A type missing from here counts as 1.

type_weights = {
    'pyod anomaly detection': 1,
    'kmeans anomaly detection': 1,
    'dbscan anomaly detection': 1,
    'Specification Based Detection': 1,
    'Signal Fusion: Model Convergence': 2,
    'Signal Fusion: ML and Conventional Alert': 2,
}

def eni_score(types):
    counts = types.value_counts()
    diversity = sum(type_weights.get(t, 1) * np.log1p(c) for t, c in counts.items())
    volume = np.log1p(len(types))
    return diversity * volume

atomic_weight = (
    sp
    .groupby('interface-id')['type']
    .apply(eni_score)
    .rename('atomic weight')
    .round(1)
)

atomic_mass = sp.groupby('interface-id').size().rename('atomic mass')

combined = atomic_weight * atomic_mass
k = combined.max()
scores = (1000 * np.tanh(combined / k)).rename('score').round().astype(int)

# signal fusion per ENI: whether it has any Signal Fusion rows (cell 6), and how many of its
# sessions (distinct session_id) have one -- a session with both fusion types counts once
_fus = sp[sp['type'].astype(str).str.startswith('Signal Fusion')]
fusion_sessions = (_fus.groupby('interface-id')['session_id'].nunique()
                   .reindex(atomic_mass.index, fill_value=0).rename('fusion sessions'))
has_fusion = fusion_sessions.gt(0).map({True: 'yes', False: 'no'}).rename('signal fusion')

# rows of ENIs with signal fusion are highlighted red; no other highlighting
FUSION_ROW_STYLE = 'background-color: #c00000; color: white; font-weight: bold'

threat_cmap = LinearSegmentedColormap.from_list('threat', ['#FFD700', '#FF6600', '#CC0000', '#888888'])

(
    pd.concat([atomic_weight, atomic_mass, scores, has_fusion, fusion_sessions], axis=1)
    .sort_values('score', ascending=False)
    .style
    .format({'atomic weight': '{:.1f}', 'fusion sessions': '{:,}'})
    .set_table_styles([{'selector': 'th', 'props': 'text-align: left; padding: 4px 12px;'},
                       {'selector': 'td', 'props': 'padding: 4px 12px;'}])
    .apply(lambda r: [FUSION_ROW_STYLE if r['signal fusion'] == 'yes' else ''] * len(r), axis=1)
)

In [ ]:
# 10 - viewer

ALL_FIELDS = [c for c in sp.columns if c != 'interface-id']
DEFAULT_FIELDS = {'type', 'tag-service', 'tag-direction', 'tag-country', 'tag-asn'}

eni_dropdown = widgets.Dropdown(
    options=sorted(sp['interface-id'].dropna().unique()),
    description='ENI:',
    layout=widgets.Layout(width='600px'),
)

select_all = widgets.Checkbox(value=False, description='Select all', indent=False,
                              layout=widgets.Layout(width='140px'))

field_checkboxes = {
    col: widgets.Checkbox(value=(col in DEFAULT_FIELDS), description=col, indent=False,
                          layout=widgets.Layout(width='220px'))
    for col in ALL_FIELDS
}

html_out = widgets.HTML()
_updating = False

def refresh(_=None):
    fields = [col for col, cb in field_checkboxes.items() if cb.value]
    if not fields:
        html_out.value = '<i>Select at least one field.</i>'
        return
    eni = eni_dropdown.value
    subset = (
        sp[sp['interface-id'] == eni][fields]
        .value_counts()
        .reset_index()
        .rename(columns={0: 'count'})
        .sort_values(fields)
    )
    html_out.value = (
        subset.style.hide(axis='index')
        .set_table_styles([{'selector': 'th', 'props': 'text-align: left; padding: 4px 12px;'},
                            {'selector': 'td', 'props': 'padding: 4px 12px;'}])
        .background_gradient(subset=['count'], cmap=threat_cmap)
        ._repr_html_()
    )

def on_select_all(change):
    global _updating
    if _updating:
        return
    _updating = True
    for cb in field_checkboxes.values():
        cb.value = change['new']
    _updating = False
    refresh()

def on_field_change(_=None):
    if _updating:
        return
    refresh()

eni_dropdown.observe(refresh, names='value')
select_all.observe(on_select_all, names='value')
for cb in field_checkboxes.values():
    cb.observe(on_field_change, names='value')

refresh()
rows = [widgets.HBox(list(field_checkboxes.values())[i:i+4]) for i in range(0, len(ALL_FIELDS), 4)]
display(eni_dropdown, widgets.HBox([select_all]), *rows, html_out)